In [39]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


In [17]:
URL = 'https://www.wizzair.com/en-gb/flights/fare-finder/skopje/anywhere/0/0/0/1/0/0/2025-12-07/2025-12-31?flexible=anytime&duration=1_week'

In [45]:
options = Options()
options.add_argument("--headless")  # optional if you want to see the browser
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")

driver = webdriver.Chrome(options=options)

In [48]:

try:
    driver.get(URL)

    wait = WebDriverWait(driver, 15)  

    flight_locations = wait.until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "input[data-test='search-arrival-station']"))
    )

    flight_locations.click()
    
    country_groups = wait.until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div[role='group'][aria-label]"))
    )
    
    all_airports = []
    
    for country in country_groups:
        country_name = country.get_attribute("aria-label")
    
        # Find all airport options in this country
        airports = country.find_elements(By.CSS_SELECTOR, "div[role='option'][value][aria-label]")
        for airport in airports:
            airport_name = airport.get_attribute("aria-label")
            airport_code = airport.get_attribute("value")
    
            # Skip "All Airports"
            if "All Airports" in airport_name:
                continue
    
            all_airports.append({
                "country": country_name,
                "airport_name": airport_name,
                "airport_code": airport_code
            })
    
    # Print results
    for a in all_airports:
        print(a)

finally:
    driver.quit()


MaxRetryError: HTTPConnectionPool(host='localhost', port=34231): Max retries exceeded with url: /session/3dec997939478eba8b5bec43d12c2391/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x00000241C322F100>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))

In [5]:
import re
import shutil
import tempfile
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait


WIZZ_AIR_HOME_URL = "https://www.wizzair.com/en-gb"


def get_base_url():
    options = Options()

    # Required on Render/Linux servers
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")

    # Prevent Chrome profile/permission issues on cloud environments
    user_data_dir = tempfile.mkdtemp(prefix="chrome-user-data-")
    options.add_argument(f"--user-data-dir={user_data_dir}")

    # Optional but often useful
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-background-networking")
    options.add_argument("--disable-default-apps")

    driver = None

    try:
        driver = webdriver.Chrome(options=options)

        driver.get(WIZZ_AIR_HOME_URL)

        WebDriverWait(driver, 30).until(
            lambda d: "apiUrl" in d.page_source or "Human Verification" in d.page_source
        )

        html = driver.page_source

        if "Human Verification" in html:
            raise Exception("Blocked by Wizz Air / AWS WAF human verification")

        match = re.search(r'apiUrl\s*:\s*"([^"]+)"', html)

        if not match:
            raise Exception("Wizz Air apiUrl not found in page source")

        api_base_url = match.group(1)
        print(api_base_url)

        return f"{api_base_url}/search/timetable"

    finally:
        if driver:
            driver.quit()

        shutil.rmtree(user_data_dir, ignore_errors=True)